# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

> 이 내용을 `chapter10.ipynb`의 Markdown 셀로 작성합니다. 실제 실행 결과를 근거로 작성하고, 예측 패턴을 원인처럼 단정하지 않습니다.

## 제출 정보
- 이름: 최종인
- GitHub ID: JONG-IN-01
- 작성일: 2026-11-16
- 최종 Notebook URL:https://github.com/JONG-IN-01/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb

## 1. 분류 문제 정의
- 타깃: 주문 취소 여부
- 0의 의미: 정상적으로 완료된 주문
- 1의 의미: 취소된 주문
- 제외한 상태: refunded건
- 예측 단위: 주문 1건
- 예측 시점: 주문이 접수된 직후
- 예측 시점에 사용할 수 있는 정보: 고객정보, 주문정보, 주문상품정보
- 예측 이후에만 알 수 있어 제외한 정보: order_status, is cancelled, cancel_reason, cancelled_at

### 나의 해석과 판단
refunded는 주문이 완료된 뒤 반품 환불된 경우로 볼 수 있어, 주문단계에서 철회하는 cancelled와 성격이 다르다. 이를 완료로 쓰면 완료의 의미가 약해지고 취소에 섞으면 서로 다른 현상을 같이 하나로 예측하게 된다. 그래서 범위를 취소와 완료로만 좁힌다.

---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature
'age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup'
### 사용한 범주형 Feature
'gender', 'city', 'payment_method'
### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보
```

### 나의 해석과 판단
파일에 칼럼이 있다고 해서 예측하는 순간에 그 값을 알 수 있는 것이 아니다. order_id, customer_id, product_id 같은 식별자는 번호일 뿐 의미 있는 패턴이 없어 제외했다. 
---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수: 0
- 주문 특징 `order_id` 중복 건수: 0

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one |203|203|0|PASS|
| 주문 ↔ 고객 | many_to_one |203|203|0|PASS|

### 결과 관찰
두 병합 모두 병합 전후 행의 수가 203행으로 같고 미매칭이 0건이다. 병합과정에서 행이 사라지거나 추가되지 않았다. 다만 그 전에 제가 오류로 인해 고객 가입일(signup_date)이 주문일(order_date)보다 늦은 주문이 56건(분류대상 45건)발견되어, 이를 오류로 보고 주문과 주문상세에서 제외했습니다.(클로드로 코드를 짰습니다.)

### 나의 해석과 판단
미매칭은 연결되지 않은 정보가 없다는 뜻이다. 주문상세가 없는 주문을 0으로 채우면 실제로 존재하지 않는 0원짜리 주문이 될 수 있다. 이와 같은 경우 먼저 원인을 살펴보고 수정 제외 여부를 결정해야한다.

---

## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed |149 |73.4%|
| 1 | cancelled |54|26.6%|

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰
완료주문이 취소주문보다 약 2.8배 많다. 클래스 불균형이라고 볼 수 있다. 

### 나의 해석과 판단
모든 주문을 완료라고 찍으면 약 73%는 맞게 나온다. 이는 실제로 모델이 취소를 잘 찾는지 판단할 수 없을 것이다.

---

## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent |0.732|0|0|0|
| Logistic Regression |0.634|0.4|0.727|0.516|
| Random Forest |0.732|0.5|0.273|0.353|

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델: Logistic Regression

### 결과 관찰
Random Forest는 accuracy가 Dummy와 같은 0.732지만 취소 11건 중 3건만 찾았다. Logistic Regression은 accuracy는 0.634로 제일 낮았지만 취소 11건 중 8건을 찾았다.

### 나의 해석과 판단
Dummy는 취소를 전혀 예측하지 못한 반면, Logistic Regression은 일부 완료주문을 취소로 잘못 예측하는 대가로 실제 취소주문을 찾아냈다.

### 한계와 추가 확인 사항
validation 데이터는 불균형 데이터이다. 한번의 random 결과이므로 교차검증 등으로 결과의 보완이 필요하다.

---

## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 학습 |89|32|
| Validation | 모델·Threshold 선택 |30|11|
| Test | 선택 이후 최종 평가 |30|11|

### 나의 해석과 판단
test는 처음 보는 데이터에 적용해야한다. test 점수를 보면서 모델이나 threshold를 고르면 Test에 맞춰 선택한 꼴이 된다. 그래서 Validation으로 선택하고, Test는 고정한 뒤 사용해야한다.

### 교육용 random split의 한계
기간을 과거 데이터로 학습하고 미래 데이터를 예측하는 것으로 해야한다. 여러 단계로 반복 검증해야한다.
---

## 7. Validation 성능과 우선 지표

- Accuracy: 0.634
- Precision: 0.409
- Recall: 0.818
- F1: 0.545

### 내가 가장 중요하게 본 지표
recall

### 그 이유
실제 FP/FN 비용 정보는 없다. 다만 취소를 놓치면(FN) 재고 확보 배송 준비 비용 등이 그대로 낭비되고, 완료주문을 취소 위험으로 잘못 분류하면(FP) 확인 연락정도의 비용이든다. 이 가정에서는 취소 주문을 잘 찾는게 중요하다.  

---

## 8. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
|0.3|0.310|0.818|0.450|취소를 넓게 잡아 놓치는건 적지만, 잘못된 취소 경보가 많음|
|0.45|0.409|0.818|0.545|같은 recall을 유지하면서 precision이 높아져 f1이 높다.|
|0.70|0.8|0.364|0.5|확실한 경우에만 취소로 예측하도록 해서 정확하지만 취소의 2/3 정도를 놓침|

![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold
0.45
### 선택 이유
validation에서 F1이 0.545로 가장 높았다. 0.6도 f1이 0.545로 같았지만 recall이 0.818로 높아 취소를 더 많이 찾는 0.45로 선택했다.
### 나의 해석과 판단
test 전에 threshold를 선택해야 test가 처음보는 데이터로 하는 것이 된다.
---

## 9. FP/FN 해석

- FP 의미: 실제로는 완료될 주문을 취소로 잘못 예측
- FN 의미: 실제로는 취소될 주문을 완료로 잘못 예측
- 현재 목적에서 더 부담스러울 수 있는 오류: 취소를 놓치면 재고 배송 준비가 낭비되고 사전 대응 기회를 잃게된다.
- 그 판단에 추가로 필요한 업무 정보: 취소 1건당 손실, 잘못된 경보가 고객에게 미치는 영향

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미
test에서 실제 완료 주문 30건 중 17건을 취소로 잘못 예측했다. 실제 취소 주문도 11건 중 4건을 찾았다. 이 모델로 하면 제대로 작업을 할 수 없다.
### 한계와 추가 확인 사항
FP/FN 중 어느쪽이 손실이 큰 지는 가정이다. 실제 비용을 확인해야한다.
---

## 10. Final Test

- 고정한 모델: Logistic Regression
- 고정한 Threshold: 0.45
- Accuracy: 0.415
- Precision: 0.190
- Recall: 0.364
- F1: 0.250
- Validation 대비 변화: 모든 지표에서 크게 하락했다.

### 결과 관찰
Test의 성능이 Validation의 절반 이하로 떨어졌다. 

### 나의 해석과 판단
Validation에서 좋게 나온 성능은 일반화된 패턴 보다는 작은 표본에서 생긴 우연일 가능성이 있다. 따라서 일반화 가능성은 낮다.

### 선택 보호 확인
- [O] Test 결과를 보기 전에 모델을 고정했습니다.
- [O] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [O] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

---

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰
test001처럼 record_id를 사용했다.

### 나의 해석과 판단
내부자료는 원본 데이터로 거슬러 올라가 오류 원인을 분석하는데 필요하다. 하지만 같은 정보가 공개되면 개인정보 노출 위험이 생긴다. 그래서 분석용과 공유용을 나누고, 공개본에는 최소정보만 남긴다. 

---

## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [O] `ch10_target_distribution.csv`
- [O] `ch10_feature_audit.csv`
- [O] `ch10_merge_checks.csv`
- [O] `ch10_data_quality_checks.csv`
- [O] `ch10_split_summary.csv`
- [O] `ch10_validation_model_comparison.csv`
- [O] `ch10_validation_threshold_metrics.csv`
- [O] `ch10_test_metrics.csv`
- [O] `ch10_confusion_matrix.csv`
- [O] `ch10_classification_validation.csv`

### 자동 Validation 결과
7개 항목 모두 PASS다. 

### 자동 PASS가 의미하지 않는 것
PASS라는 것은 절차를 규칙대로 지켰다는 것이다. 모델이 쓸만하다라는 것은 아니다. 운영적합성, 공정성, 시간이 지나면 유지되는지 등 그 외의 정보는 파악할 수 없다.

---

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [O] 현재 사용 보류

### 판단 근거
1. f1이 0.250 accuracy가 0.415로 dummy보다 낮다.
2. 평가표본이 41건으로 작다.
3. 가입일보다 앞선 주문이 56개가 있어 삭제되어 올바른 검증이 안될 수 있다.

### 현재 모델의 가장 큰 위험
validation 성능만 보고 취소를 82%로 잡는 모델로 오해할 수 있다.
실제 업무에는 사용하면 82% 성능이 안 나올 수 있다.

### 다음 개선 우선순위
1. 데이터의 품질과 양 확보
2. 평가방식의 강화 - 예 : 시간기준 분할과 교차검증으로 성능이 우연인지 확인 필요
3. FP/FN 실제 비용을 계산해 threshold를 다시 정한다.

### 예측과 인과 구분
이 모델이 보여주는 것은 어떤 특징을 가진 주문이 취소 쪽으로 분류되는 경향이 있다고 예측하는 것이다.
특정 원인 때문에 취소가 일어난다는 주장은 할 수 없다.
---

## 최종 체크

- [O] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [O] `refunded`와 기타 상태를 제외했습니다.
- [O] 예측 시점과 Feature Contract를 정의했습니다.
- [O] target, ID, 사후 정보를 feature에서 제외했습니다.
- [O] `line_total = quantity × unit_price`를 검증했습니다.
- [O] 주문 단위 특징과 병합 관계를 검증했습니다.
- [O] 클래스 비율을 확인했습니다.
- [O] Dummy baseline과 비교했습니다.
- [O] Train / Validation / Test 역할을 구분했습니다.
- [O] 모델을 Validation에서 선택했습니다.
- [O] Threshold를 Validation에서 선택했습니다.
- [O] Final Test 전에 선택을 고정했습니다.
- [O] Precision / Recall / F1을 해석했습니다.
- [O] FP/FN의 의미를 설명했습니다.
- [O] Public prediction에서 식별자를 제거했습니다.
- [O] Validation Evidence를 확인했습니다.
- [O] random split의 한계를 기록했습니다.
- [O] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [O] 최종 Notebook URL을 제출합니다.

## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [6]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


Python 실행 파일: c:\dev\llm-data-analysis-study\.venv\Scripts\python.exe
프로젝트 루트: C:\dev\llm-data-analysis-study
전처리 데이터 폴더: C:\dev\llm-data-analysis-study\data\processed
보고서 폴더: C:\dev\llm-data-analysis-study\reports


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [7]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [8]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


아래 셀은 오류가 발생하여, 이를 해결하기 위해 클로드가 코드를 짜줘 실행한 것입니다.
클로드가 내린 오류의 원인은 "스크립트가 "가입일보다 주문일이 빠른지"를 아예 검사하지 않아서 clean 파일에도 문제가 남은 거예요."라고 하였습니다.
오류는 ValueError: signup_date가 order_date보다 늦은 주문이 있습니다: 45건
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings... 이와 같았습니다.

아래 코드를 실행하니 오류는 해결되었습니다. 
또한, 새폴더에 저장하여 기존 터미널로 만든 processed 파일은 건드리지 않았습니다.

In [9]:
# 1) 주문에 고객 가입일을 붙여서 '가입 전 주문' 찾기
check = orders.merge(
    customers[['customer_id', 'signup_date']],
    on='customer_id',
    how='left',
)
is_bad = pd.to_datetime(check['signup_date']) > pd.to_datetime(check['order_date'])
bad_order_ids = check.loc[is_bad, 'order_id']

print('가입 전 주문 수(전체 상태):', len(bad_order_ids))

# 2) 문제 주문과 그 주문의 상품 내역을 함께 빼기 (~ 는 '아닌 것')
orders = orders[~orders['order_id'].isin(bad_order_ids)].copy()
order_items = order_items[~order_items['order_id'].isin(bad_order_ids)].copy()

print('정리 후 orders:', orders.shape)
print('정리 후 order_items:', order_items.shape)

# 3) 마지막 셀에서 쓸 정리본을 새 폴더에 저장 (원래 processed 파일은 그대로 둠)
CLEAN_DIR = PROJECT_ROOT / 'data' / 'processed_ch10'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
customers.to_csv(CLEAN_DIR / 'customers_clean.csv', index=False)
orders.to_csv(CLEAN_DIR / 'orders_clean.csv', index=False)
order_items.to_csv(CLEAN_DIR / 'order_items_clean.csv', index=False)

가입 전 주문 수(전체 상태): 56
정리 후 orders: (244, 7)
정리 후 order_items: (624, 6)


In [10]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


모델링 데이터: (203, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,149,0.734
1,1,cancelled,54,0.266


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,203,203,True,0,PASS
1,customer_id → customers,203,203,True,0,PASS


,check_item,count
0,binary_target_rows,203
1,completed_rows,149
2,cancelled_rows,54
3,excluded_status_rows,41
4,line_total_mismatch,0
5,negative_days_since_signup,0
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,149
9,status_scope:cancelled,54


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [11]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [12]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (121, 10)
X_valid: (41, 10)
X_test: (41, 10)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,89,0.7355
1,train,1,cancelled,32,0.2645
2,validation,0,completed,30,0.7317
3,validation,1,cancelled,11,0.2683
4,test,0,completed,30,0.7317
5,test,1,cancelled,11,0.2683


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [13]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.634146,0.4,0.727273,0.516129
1,Random Forest,validation,0.731707,0.5,0.272727,0.352941
2,Dummy Most Frequent,validation,0.731707,0.0,0.000000,0.000000


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [14]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


선택 모델: Logistic Regression
선택 임계값: 0.45


,threshold,accuracy,precision,recall,f1
5,0.45,0.634146,0.409091,0.818182,0.545455
8,0.60,0.756098,0.545455,0.545455,0.545455
7,0.55,0.707317,0.466667,0.636364,0.538462
6,0.50,0.634146,0.400000,0.727273,0.516129
4,0.40,0.585366,0.375000,0.818182,0.514286
10,0.70,0.804878,0.800000,0.363636,0.500000
3,0.35,0.536585,0.346154,0.818182,0.486486
9,0.65,0.780488,0.666667,0.363636,0.470588
2,0.30,0.463415,0.310345,0.818182,0.450000
0,0.20,0.341463,0.289474,1.000000,0.448980


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [15]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.45,frozen_before_test,0.414634,0.190476,0.363636,0.25


,pred_completed,pred_cancelled
actual_completed,13,17
actual_cancelled,7,4


,label,precision,recall,f1-score,support
0,completed,0.650000,0.433333,0.520000,30.000000
1,cancelled,0.190476,0.363636,0.250000,11.000000
2,accuracy,0.414634,0.414634,0.414634,0.414634
3,macro avg,0.420238,0.398485,0.385000,41.000000
4,weighted avg,0.526713,0.414634,0.447561,41.000000


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [16]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()


,record_id,source_index,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,79,0,1,0.462696,Logistic Regression,0.45
1,test_0002,20,0,1,0.498074,Logistic Regression,0.45
2,test_0003,104,0,0,0.391324,Logistic Regression,0.45
3,test_0004,119,1,1,0.946596,Logistic Regression,0.45
4,test_0005,29,0,1,0.482105,Logistic Regression,0.45


## 11. LLM 코드 검토 체크리스트


In [17]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist


,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,□
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,□
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,□
3,line_total = quantity × unit_price를 검증했는가?,□
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,□
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",□
6,"train, validation, test를 분리했는가?",□
7,모델과 threshold는 validation에서 선택했는가?,□
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,□
9,Dummy 기준 모델과 비교했는가?,□


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [18]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=CLEAN_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


선택 모델: Logistic Regression
선택 임계값: 0.45


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.45,frozen_before_test,0.414634,0.190476,0.363636,0.25


model_data_internal C:\dev\llm-data-analysis-study\reports\ch10_classification_model_data_internal.csv OK
target_distribution C:\dev\llm-data-analysis-study\reports\ch10_target_distribution.csv OK
feature_audit C:\dev\llm-data-analysis-study\reports\ch10_feature_audit.csv OK
merge_checks C:\dev\llm-data-analysis-study\reports\ch10_merge_checks.csv OK
data_quality_checks C:\dev\llm-data-analysis-study\reports\ch10_data_quality_checks.csv OK
split_summary C:\dev\llm-data-analysis-study\reports\ch10_split_summary.csv OK
validation_comparison C:\dev\llm-data-analysis-study\reports\ch10_validation_model_comparison.csv OK
threshold_metrics C:\dev\llm-data-analysis-study\reports\ch10_validation_threshold_metrics.csv OK
test_metrics C:\dev\llm-data-analysis-study\reports\ch10_test_metrics.csv OK
predictions_internal C:\dev\llm-data-analysis-study\reports\ch10_classification_predictions_internal.csv OK
predictions C:\dev\llm-data-analysis-study\reports\ch10_classification_predictions.csv OK
con

## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.
